# FA 점수체계 자동화 및 검증

이 노트북은 설명용 문서가 아니라 실행용 검증 노트북이다.

자세한 개념은 `FA_학습가이드.md`와 `FA_전략.md`를 먼저 본다.

## 목표

검증 순서:

```text
1. 데이터 로드
2. 한 섹터 선택
3. 점수 분포 확인
4. 상위/하위 종목 확인
5. 결측과 신뢰도 확인
6. 이후 수익률 검증으로 확장
```

운영 목표는 사람이 매번 노트북을 열어 판단하는 것이 아니다. 이 노트북은 자동화 스크립트가 만든 결과를 사람이 필요할 때 점검하는 보조 화면이다.

자동화의 기본 방향은 하방 방어다.

```text
pass    → 신규 결과 사용
warning → 마지막 정상 결과 유지
fail    → 결과 격리 + 마지막 정상 결과 유지
```

즉 애매하거나 실패한 점수를 새로운 리밸런싱 입력으로 밀어 넣지 않는다.

이렇게 정의하는 이유는 FA 점수가 백테스트와 리밸런싱의 입력값이기 때문이다. 새 점수를 못 쓰는 것보다 더 위험한 것은, 신뢰하기 어려운 새 점수를 정상 결과처럼 사용하는 것이다.

전략 문서의 최신 점수 계산 구조는 아래와 같다.

```text
원천 재무값
→ 파생 지표 계산
→ 같은 분기/섹터 내 percentile 변환
→ level_score, change_score, risk_penalty 계산
→ score_confidence 계산
→ fa_score와 fallback 판정
```

이 노트북의 현재 데이터는 연간 임시 랭킹이므로 위 구조를 완전히 구현하지는 않는다. 대신 자동화 검증 화면이 어떤 항목을 확인해야 하는지 먼저 정의한다.

현재는 `company_quarter_fa` 구현 전이므로 기존 `company_sector_rankings_2021_2025.csv`를 임시 검증 데이터로 사용한다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('..').resolve()
RANKINGS_PATH = PROJECT_ROOT / 'etl' / 'wics_dart' / 'output' / 'company_sector_rankings_2021_2025.csv'

rankings = pd.read_csv(RANKINGS_PATH, dtype={'stock_code': str})
rankings.head()

## 한 섹터 선택

처음에는 하나의 섹터만 본다. 여러 섹터를 한 번에 보면 어떤 기준이 문제인지 찾기 어렵다.

In [ ]:
SAMPLE_SECTOR = 'IT'
SAMPLE_YEAR = 2025

sample = rankings[
    (rankings['wics_large'] == SAMPLE_SECTOR)
    & (rankings['fiscal_year'] == SAMPLE_YEAR)
].copy()

sample.shape

## 점수 분포 확인

좋은 점수체계라면 상위, 중위, 하위가 어느 정도 나뉘어야 한다. 모든 종목이 비슷한 점수에 몰리면 변별력이 부족하다.

In [ ]:
score_summary = pd.DataFrame([{
    'sector': SAMPLE_SECTOR,
    'fiscal_year': SAMPLE_YEAR,
    'rows': len(sample),
    'score_count': sample['overall_score'].notna().sum(),
    'score_missing': sample['overall_score'].isna().sum(),
    'score_mean': sample['overall_score'].mean(),
    'score_std': sample['overall_score'].std(),
    'score_min': sample['overall_score'].min(),
    'score_max': sample['overall_score'].max(),
}])

score_summary

## 상위/하위 종목 확인

점수가 좋은지 보려면 상위 종목과 하위 종목의 이유가 설명되어야 한다.

In [ ]:
review_columns = [
    'company_name',
    'stock_code',
    'overall_score',
    'overall_bucket',
    'growth_score',
    'profitability_score',
    'stability_score',
    'cashflow_score',
    'shareholder_return_score',
]

top_10 = sample.nlargest(10, 'overall_score')[review_columns]
top_10

In [ ]:
bottom_10 = sample.nsmallest(10, 'overall_score')[review_columns]
bottom_10

## 결측과 신뢰도 확인

상위 종목에 결측이 많으면 점수를 그대로 믿으면 안 된다. 그래서 `score_confidence`가 필요하다.

In [ ]:
score_columns = [
    'growth_score',
    'profitability_score',
    'stability_score',
    'cashflow_score',
    'shareholder_return_score',
]

missing_in_top = (
    top_10[score_columns]
    .isna()
    .sum()
    .rename('missing_count_in_top_10')
    .reset_index()
    .rename(columns={'index': 'score_column'})
)

missing_in_top

In [ ]:
sample['score_confidence_temp'] = sample[score_columns].notna().mean(axis=1)

sample.nlargest(10, 'overall_score')[
    ['company_name', 'stock_code', 'overall_score', 'score_confidence_temp']
]

## 자동 판정 정책

운영 자동화에서는 사람이 매번 상위/하위 종목을 보고 판단하지 않는다. 노트북은 자동 판정 결과를 설명하는 보조 화면으로 둔다.

v1 기본 정책은 하방 방어다.

이 정책은 수익 기회를 더 많이 잡기 위한 규칙이 아니라, 검증되지 않은 점수가 다음 단계로 흘러 들어가는 것을 막기 위한 규칙이다.

| 판정 | 기본 행동 |
|---|---|
| `pass` | 신규 결과 사용 |
| `warning` | 마지막 정상 결과 유지 |
| `fail` | 결과 격리 + 마지막 정상 결과 유지 |

검증 변수는 아래처럼 원천 컬럼에서 계산된다.

| 변수 | 계산 출처 | 의미 |
|---|---|---|
| `score_missing_rate` | `sample['overall_score'].isna().mean()` | 점수 자체가 생성되지 않은 종목 비율 |
| `score_std` | `sample['overall_score'].std()` | 점수 분포의 변별력 |
| `score_confidence_temp` | `score_columns`의 결측 여부 | 점수가 실제 사용 가능한 하위 지표를 얼마나 기반으로 하는지 |
| `top_confidence_min` | 상위 10개 종목의 `score_confidence_temp` 최솟값 | 상위 종목이 결측 때문에 왜곡되지 않았는지 |

현재 `score_confidence_temp`는 임시 계산이다. 실제 자동화에서는 `score_model`별로 실제 산식에 들어간 하위 지표만 사용해야 한다.

아래 셀은 현재 임시 데이터 기준으로 자동 판정 예시를 만든다. 실제 분기 자동화에서는 같은 로직을 스크립트와 설정 파일로 옮긴다.

In [ ]:
validation_config = {
    'max_score_missing_rate': 0.25,
    'min_score_std': 0.05,
    'min_top_score_confidence': 0.70,
}

# 변수 출처
# - score_missing_rate: overall_score 결측률
# - score_std: overall_score 분포의 표준편차
# - top_confidence_min: 상위 10개 종목의 score_confidence_temp 최솟값
# 실제 분기 자동화에서는 이 기준값을 설정 파일로 분리한다.

score_missing_rate = sample['overall_score'].isna().mean()
score_std = sample['overall_score'].std()
top_confidence_min = (
    sample.nlargest(10, 'overall_score')['score_confidence_temp'].min()
    if sample['overall_score'].notna().any()
    else 0
)

checks = pd.DataFrame([
    {
        'check': 'score_missing_rate',
        'value': score_missing_rate,
        'threshold': validation_config['max_score_missing_rate'],
        'status': 'pass' if score_missing_rate <= validation_config['max_score_missing_rate'] else 'fail',
    },
    {
        'check': 'score_std',
        'value': score_std,
        'threshold': validation_config['min_score_std'],
        'status': 'pass' if score_std >= validation_config['min_score_std'] else 'warning',
    },
    {
        'check': 'top_score_confidence_min',
        'value': top_confidence_min,
        'threshold': validation_config['min_top_score_confidence'],
        'status': 'pass' if top_confidence_min >= validation_config['min_top_score_confidence'] else 'warning',
    },
])

if (checks['status'] == 'fail').any():
    validation_status = 'fail'
elif (checks['status'] == 'warning').any():
    validation_status = 'warning'
else:
    validation_status = 'pass'

fallback_action = {
    'pass': 'use_new_result',
    'warning': 'hold_previous',
    'fail': 'quarantine_and_hold_previous',
}[validation_status]

validation_decision = pd.DataFrame([{
    'sector': SAMPLE_SECTOR,
    'fiscal_year': SAMPLE_YEAR,
    'validation_status': validation_status,
    'fallback_action': fallback_action,
}])

checks, validation_decision

## 이번 샘플의 판단

이 절은 사람이 최종 결정을 내리는 부분이 아니라, 위 자동 판정 결과를 해석하는 부분이다.

2025년 IT 섹터 샘플 참고값:

| 항목 | 값 |
|---|---:|
| 전체 종목 수 | 709 |
| 점수 존재 종목 수 | 592 |
| 평균 점수 | 0.4998 |
| 표준편차 | 0.1987 |
| 최소 점수 | 0.0276 |
| 최대 점수 | 0.9086 |

판단:

- 점수 분포는 상위/하위가 나뉘므로 변별력은 있다.
- 다만 상위 종목에도 일부 하위 점수 결측이 있어 신뢰도 검증이 필요하다.
- 따라서 운영 자동화에서는 검증 규칙에 따라 `pass`, `warning`, `fail`을 자동 판정한다.
- `warning` 또는 `fail`이면 신규 결과를 바로 쓰지 않고 하방 방어 정책을 적용한다.
- 다음 검증은 상위 20%와 하위 20%의 forward return 비교다.

## 다음에 추가할 검증

```text
1. 분기형 company_quarter_fa 생성
2. operating_margin, roe, debt_ratio 같은 파생 지표 계산
3. available_date 기준 시점 검증
4. level_score, change_score, risk_penalty 계산 결과 검증
5. score_model별 실제 사용 지표 기준으로 score_confidence 계산
6. pass/warning/fail 자동 판정 규칙을 설정 파일로 분리
7. warning/fail 시 hold_previous 또는 quarantine 정책 적용
8. 섹터별 상위 20% / 하위 20% 구분
9. 1개월, 3개월, 6개월 forward return 비교
10. 자동화 스크립트의 산출물 점검 화면으로 노트북 역할 축소
11. vectorBT 입력으로 연결
```